# 🚀 AlphaLOB: Multi-Task LOBTransformer Training Pipeline

This notebook trains the PyTorch model on Google Colab's free T4 GPU. Once training is complete, the weights are exported to a lightweight ONNX format to be downloaded to your local 8GB RAM laptop.

In [ ]:
# 1. Setup Environment
!pip install torch==2.2.0 flash-attn hmmlearn polars
!pip install onnx onnxruntime

In [ ]:
# 2. Clone the Repository (so Colab has access to the architecture files)
import sys
import os

!git clone https://github.com/yourusername/AlphaLOB.git
sys.path.append(os.path.abspath('AlphaLOB'))

In [ ]:
# 3. Load Data & Extract Features
# (In a real scenario, you'd mount Google Drive and load LOBSTER data here)
import torch
import numpy as np
import polars as pl
from src.domain.models.lob_transformer import LOBTransformer
from src.domain.models.multi_task_head import AlphaLOBModel
from src.domain.models.regime_hmm import RegimeHMM

print("Generating synthetic training batch for demonstration...")
# Shape: [batch_size, n_levels, features_per_level]
# 10 levels, 4 features (bid_price, bid_vol, ask_price, ask_vol)
batch_size = 1024
dummy_X = torch.randn(batch_size, 10, 4).cuda()
dummy_y_dir = torch.randint(0, 2, (batch_size, 1)).float().cuda()
dummy_y_spread = torch.randint(0, 2, (batch_size, 1)).float().cuda()
dummy_y_vol = torch.randn(batch_size, 1).cuda()

In [ ]:
# 4. Initialize Multi-Task Model & Optimizer
model = AlphaLOBModel(n_levels=10, features_per_level=4, d_model=64).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
bce_loss = torch.nn.BCELoss()
mse_loss = torch.nn.MSELoss()

print("Starting 1 epoch training loop...")
model.train()
optimizer.zero_grad()

dir_5s, dir_30s, dir_5m, spread_comp, vol_imb = model(dummy_X)

# Multi-task loss (simple sum for demo, in production use Kendall uncertainty weighting)
loss = bce_loss(dir_5s, dummy_y_dir) + \
       bce_loss(spread_comp, dummy_y_spread) + \
       mse_loss(vol_imb, dummy_y_vol)

loss.backward()
optimizer.step()
print(f"Training Loss: {loss.item():.4f}")

In [ ]:
# 5. Train Regime HMM
print("Training Hidden Markov Model for volatility regimes...")
hmm = RegimeHMM(n_states=3)
# [realized_vol, autocorrelation] dummy features
hmm_features = np.random.randn(5000, 2) 
hmm.fit(hmm_features)
hmm.save('regime_hmm.pkl')

In [ ]:
# 6. Export to ONNX
print("Exporting trained model to ONNX format...")
model.eval()
model.cpu()
dummy_input_cpu = torch.randn(1, 10, 4)

torch.onnx.export(
    model,
    dummy_input_cpu,
    "lobster_transformer.onnx",
    input_names=['lob_snapshot'],
    output_names=['dir_5s', 'dir_30s', 'dir_5min', 'spread_compress', 'vol_imbalance'],
    dynamic_axes={'lob_snapshot': {0: 'batch_size'}},
    opset_version=17
)
import os
print(f"ONNX Export Complete! File size: {os.path.getsize('lobster_transformer.onnx') / 1e6:.2f} MB")

In [ ]:
# 7. Download Files to Local Laptop
try:
    from google.colab import files
    files.download('lobster_transformer.onnx')
    files.download('regime_hmm.pkl')
    print("Place these files in your local AlphaLOB/src/models/weights/ directory.")
except ImportError:
    print("Not running in Colab environment. Files are saved to current directory.")